# Auditoría contra la base — mediciones que requieren conexión

Complementa a [`Auditoria.ipynb`](Auditoria.ipynb), que trabaja sobre un dataset exportado.
Las cinco mediciones de aquí **no se pueden hacer con un export**: unas necesitan datos crudos
sin preprocesar, otra compara el export contra la fuente, y otras usan tablas que el export no
incluía.

| # | Pregunta | Respuesta obtenida |
|---|---|---|
| 1 | ¿El target es estable entre extracciones? | **No** — se recalculó en todos los periodos |
| 2 | ¿Ajustar el preprocesamiento antes del split infla? | No — efecto **−0.004** |
| 3 | ¿La partición temporal infla, con datos actuales? | No — efecto **−0.006** |
| 4 | ¿Cuánto cuesta anticiparse un periodo? | **−0.128** de AUC |
| 5 | ¿La actividad en plataforma aporta? | **+0.24**, pero no verificable |

> **Prerrequisitos:** acceso de red a la base, driver ODBC, y las variables de entorno
> `DB_HOST`, `DB_PORT`, `DB_NAME` y `STATUS_DESERCION`. Todas las consultas son de **solo
> lectura**. Los nombres de esquema y tabla son genéricos: sustitúyelos por los de tu
> instalación.

## Conexión y utilidades

In [ ]:
import os
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

host, port, database = os.getenv('DB_HOST'), os.getenv('DB_PORT', '1433'), os.getenv('DB_NAME')
VALOR_DESERCION = os.getenv('STATUS_DESERCION', '').strip().lower()
if not all([host, database, VALOR_DESERCION]):
    raise RuntimeError("Define DB_HOST, DB_NAME y STATUS_DESERCION. Ver .env.example")

engine = create_engine(
    'mssql+pyodbc://@{h}:{p}/{d}?trusted_connection=yes'
    '&driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes'.format(
        h=host, p=port, d=database),
    connect_args={'timeout': 300})


def consultar(sql):
    with engine.connect() as cn:
        return pd.read_sql_query(text(sql), cn)


NUM = ['SEMESTRE_SINU', 'MATERIAS_INSCRITAS', 'MATERIAS_APROBADAS',
       'PORCENTAJE_APROBACION', 'TOTAL', 'FLAG_NO_APROBO_NADA']
CAT = ['TIPO_SALTO', 'MODALIDAD', 'GENERO', 'RANGO_EDAD', 'RANGO_SALARIO',
       'ESTA_TRABAJANDO', 'METODO_FINANCIAMIENTO', 'ZONA_RESIDENCIA',
       'REGIMEN_SISTEMA_SALUD']


def modelo(spw, num, cat):
    return Pipeline([
        ('prep', ColumnTransformer([
            ('n', Pipeline([('i', SimpleImputer(strategy='median')),
                            ('s', StandardScaler())]), num),
            ('c', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                            ('o', OrdinalEncoder(handle_unknown='use_encoded_value',
                                                 unknown_value=-1))]), cat)])),
        ('clf', LGBMClassifier(n_estimators=481, num_leaves=32, learning_rate=0.0244,
                               subsample=0.9122, random_state=42, n_jobs=-1,
                               verbose=-1, scale_pos_weight=spw))])


def precision_en_tramo(y_true, proba, pcts=(1, 5, 10, 20)):
    v = np.asarray(y_true); o = np.argsort(-proba); base = v.mean()
    return {f'P@{p}%': round(float(v[o[:int(len(v)*p/100)]].mean()), 4) for p in pcts}, base

## Carga del dataset integrado

Se deduplica cada fuente **antes** del merge y una aserción verifica que el número de filas no
cambie: sin ese control uno de los joins inflaba el dataset en silencio.

In [ ]:
SQL_BASE = '''
    SELECT Identificacion, Periodo, Status, Tipo_Salto, Modalidad, Semestre_SINU,
           año AS ANIO, Genero, RANGO_EDAD, RANGO_SALARIO, ESTA_TRABAJANDO,
           METODO_FINANCIAMIENTO, ZONA_RESIDENCIA, REGIMEN_SISTEMA_SALUD
    FROM academico.historial_academico
'''
SQL_MAT = '''
    SELECT IDENTIFICACION AS Identificacion, COD_PERIODO AS Periodo,
           [MATERIAS INSCRITAS] AS MATERIAS_INSCRITAS,
           [MATERIAS APROBADAS] AS MATERIAS_APROBADAS,
           Porcentaje_aprobacion AS PORCENTAJE_APROBACION
    FROM academico.aprobacion_materias
'''
SQL_CAR = 'SELECT Identificacion, Periodo, TOTAL FROM financiera.cartera'

base = consultar(SQL_BASE)
filas_iniciales = len(base)
for sql in (SQL_MAT, SQL_CAR):
    aux = consultar(sql).drop_duplicates(subset=['Identificacion', 'Periodo'], keep='last')
    base = base.merge(aux, on=['Identificacion', 'Periodo'], how='left')
assert len(base) == filas_iniciales, f"el merge cambió las filas: {filas_iniciales} -> {len(base)}"

base.columns = [c.strip().upper() for c in base.columns]
assert not base.columns.duplicated().any(), "hay columnas duplicadas tras normalizar nombres"

base['TARGET'] = (base['STATUS'].astype(str).str.strip().str.lower() == VALOR_DESERCION).astype(int)
base['FLAG_NO_APROBO_NADA'] = np.where(base['PORCENTAJE_APROBACION'] == 0, 1, 0)
base['PORCENTAJE_APROBACION'] = base['PORCENTAJE_APROBACION'].replace(0, np.nan)
base['ANIO'] = pd.to_numeric(base['ANIO'], errors='coerce')
base = base.dropna(subset=['ANIO', 'IDENTIFICACION'])
base['ANIO'] = base['ANIO'].astype(int)
for c in NUM: base[c] = pd.to_numeric(base[c], errors='coerce')
for c in CAT: base[c] = base[c].astype(str)

# Huella de la extracción: sin ella dos mediciones no son comparables
huella = {'fecha_utc': datetime.now(timezone.utc).isoformat(),
          'filas': len(base), 'tasa_base': round(float(base['TARGET'].mean()), 4)}
print(huella)
print("\nCobertura por variable:")
print((base[NUM].notna().mean() * 100).round(1).to_string())

> **Atención a la cobertura.** Si alguna variable aparece con 0 %, su tabla de origen está
> vacía. No producirá ningún error: el imputador la descarta y el modelo entrena con un
> predictor menos de los declarados. Ocurrió en este proyecto y pasó inadvertido en varias
> mediciones.

---
## 1. ¿Es estable el target entre extracciones?

El campo de estado se resuelve cuando se comprueba si el estudiante volvió a matricularse, así
que cabía esperar que los periodos recientes maduraran. Se compara la tasa por periodo leída
hoy contra la de un export anterior.

In [ ]:
RUTA_EXPORT = os.getenv('DATASET_AUDITORIA')   # export previo, para comparar

hoy = base.groupby('ANIO')['TARGET'].agg(['size', 'mean'])
if RUTA_EXPORT:
    xl = pd.read_excel(RUTA_EXPORT, usecols=['AÑO', 'TARGET_DESERCION'])
    xl.columns = ['ANIO', 'TARGET']
    antes = xl.groupby('ANIO')['TARGET'].agg(['size', 'mean'])
    cmp = pd.DataFrame({'tasa_export': antes['mean'].round(4),
                        'tasa_hoy': hoy['mean'].round(4)}).dropna()
    cmp['delta_pp'] = ((cmp['tasa_hoy'] - cmp['tasa_export']) * 100).round(2)
    display(cmp)
    print(f"Cambio medio en periodos cerrados hace más de 5 años: "
          f"{cmp.loc[cmp.index <= 2019, 'delta_pp'].median():+.2f} pp")
else:
    print("Define DATASET_AUDITORIA para comparar contra un export previo.")
    display(hoy.round(4))

**Resultado.** Cambiaron **todos** los periodos, incluidos los cerrados hacía una década, con
desplazamientos de entre 3 y 13 puntos porcentuales y un conteo de filas casi idéntico. No es
maduración de etiquetas —un periodo cerrado no puede madurar— sino un **recálculo del campo de
estado en el sistema origen**.

Consecuencia: las métricas medidas sobre extracciones distintas no son comparables, y la
inestabilidad del dato puede imitar la firma de una fuga de información.

---
## 2. ¿Ajustar el preprocesamiento antes del split infla el resultado?

Imposible de medir con un export: ese archivo ya trae el escalado y la imputación aplicados
sobre el total de los datos, con la fuga incorporada.

In [ ]:
tr = base[base['ANIO'] == base['ANIO'].max() - 1]
te = base[base['ANIO'] == base['ANIO'].max()]
spw = (tr['TARGET'] == 0).sum() / max((tr['TARGET'] == 1).sum(), 1)
F = NUM + CAT

# A ─ el preprocesador ve todo el dataset, incluido el periodo de evaluación
prep = ColumnTransformer([
    ('n', Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler())]), NUM),
    ('c', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                    ('o', OrdinalEncoder(handle_unknown='use_encoded_value',
                                         unknown_value=-1))]), CAT)])
todo = prep.fit_transform(base[F])
itr = (base['ANIO'] == base['ANIO'].max() - 1).to_numpy()
ite = (base['ANIO'] == base['ANIO'].max()).to_numpy()
ma = LGBMClassifier(n_estimators=481, num_leaves=32, learning_rate=0.0244, subsample=0.9122,
                    random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight=spw)
ma.fit(todo[itr], tr['TARGET'])
auc_fuga = roc_auc_score(te['TARGET'], ma.predict_proba(todo[ite])[:, 1])

# B ─ el preprocesador solo ve el entrenamiento
mb = modelo(spw, NUM, CAT); mb.fit(tr[F], tr['TARGET'])
auc_limpio = roc_auc_score(te['TARGET'], mb.predict_proba(te[F])[:, 1])

print(f"A  preprocesador ajustado con TODO     AUC {auc_fuga:.4f}")
print(f"B  preprocesador dentro del pipeline   AUC {auc_limpio:.4f}")
print(f"\nEfecto de la fuga: {auc_fuga - auc_limpio:+.4f}")

**Resultado.** La fuga **no infla**: el pipeline limpio rinde marginalmente mejor. Coherente
con el modelo elegido, que parte por umbrales y es invariante a transformaciones monótonas, de
modo que el escalado no filtra información; solo quedaba el canal de la imputación, y resultó
despreciable.

Corregirlo sigue siendo correcto por principio, pero era el menor de los problemas
sospechados.

---
## 3. ¿La partición temporal infla el AUC, con datos actuales?

Sobre el export este efecto medía **−0.101** y motivó el rediseño completo de la versión 1.8.
Se repite la medición con el target actual.

In [ ]:
sub = base[base['ANIO'] >= base['ANIO'].max() - 9]
X, y = sub[F], sub['TARGET']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
m = modelo((ytr == 0).sum() / (ytr == 1).sum(), NUM, CAT); m.fit(Xtr, ytr)
auc_rand = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])

m2 = modelo(spw, NUM, CAT); m2.fit(tr[F], tr['TARGET'])
p = m2.predict_proba(te[F])[:, 1]
auc_temp = roc_auc_score(te['TARGET'], p)

print(f"partición aleatoria  AUC {auc_rand:.4f}")
print(f"partición temporal   AUC {auc_temp:.4f}")
print(f"\nEfecto temporal: {auc_temp - auc_rand:+.4f}   (sobre el export previo: -0.1009)")

tramos, tasa = precision_en_tramo(te['TARGET'], p)
print(f"\nPrecisión por tramo priorizado: {tramos}")

**Resultado.** El efecto cae de **−0.101** a **−0.006**: desaparece. La diferencia no estaba
en el método sino en el dato, y remite al recálculo del target documentado en la sección 1.

Es la refutación más instructiva del proyecto, porque la hipótesis parecía sólida, estaba
correctamente medida sobre los datos disponibles en su momento, y aun así era un artefacto.

---
## 4. ¿Cuánto cuesta anticiparse un periodo?

Las variables pertenecen al mismo periodo que el target, de modo que la alerta se emite al
cerrar el periodo. Un sistema de alerta temprana necesita anticiparse. Se compara usando las
variables del periodo **anterior** de cada estudiante, sobre exactamente las mismas filas.

In [ ]:
panel = base.sort_values(['IDENTIFICACION', 'ANIO', 'PERIODO']).reset_index(drop=True)
prev = panel.groupby('IDENTIFICACION')[F].shift(1)
prev.columns = [f'{c}_PREV' for c in F]
panel = pd.concat([panel[['IDENTIFICACION', 'ANIO', 'TARGET'] + F], prev], axis=1)
panel = panel.dropna(subset=[f'{c}_PREV' for c in CAT], how='any')

NUM_P = [f'{c}_PREV' for c in NUM]; CAT_P = [f'{c}_PREV' for c in CAT]
ptr = panel[panel['ANIO'] == panel['ANIO'].max() - 1]
pte = panel[panel['ANIO'] == panel['ANIO'].max()]
spw2 = (ptr['TARGET'] == 0).sum() / max((ptr['TARGET'] == 1).sum(), 1)

for etiqueta, num, cat in [("A  variables del periodo t", NUM, CAT),
                           ("B  variables del periodo t-1", NUM_P, CAT_P)]:
    mm = modelo(spw2, num, cat); mm.fit(ptr[num + cat], ptr['TARGET'])
    pp = mm.predict_proba(pte[num + cat])[:, 1]
    tramos, tasa = precision_en_tramo(pte['TARGET'], pp)
    print(f"{etiqueta:<30} AUC {roc_auc_score(pte['TARGET'], pp):.4f} | {tramos}")

**Resultado.** Anticiparse un periodo cuesta **0.128 de AUC** y hunde la precisión del tramo
prioritario de 0.834 a 0.266, con un lift de 1,7×. Un síntoma lo confirma: en el modelo
desfasado el top 1 % rinde *peor* que el top 5 %, de modo que el orden se invierte y la cola
superior del ranking deja de ser informativa.

Con las variables actuales no se puede construir alerta temprana intra-periodo.

---
## 5. ¿La actividad en plataforma rescata la alerta temprana?

La fuente de actividad en la plataforma virtual registra entregas, calificaciones parciales y
participación por curso. Se agrega por estudiante y periodo y se compara contra el resto.

> Esta fuente tiene cobertura limitada a los periodos más recientes; ajusta la lista de
> periodos a lo que exista en tu instalación.

In [ ]:
PERIODOS = os.getenv('PERIODOS_PLATAFORMA', '').split(',')
if not PERIODOS or not PERIODOS[0]:
    raise RuntimeError("Define PERIODOS_PLATAFORMA con los periodos a analizar, separados por coma")
lista = ','.join(f"'{p.strip()}'" for p in PERIODOS)

plat = consultar(f'''
    SELECT num_identificacion AS IDENTIFICACION, cod_periodo AS PERIODO,
           COUNT(*)                         AS ACT_N,
           COUNT(DISTINCT curso)            AS ACT_CURSOS,
           AVG(CAST(finalgrade AS FLOAT))   AS ACT_NOTA_MEDIA,
           MIN(CAST(finalgrade AS FLOAT))   AS ACT_NOTA_MIN,
           STDEV(CAST(finalgrade AS FLOAT)) AS ACT_NOTA_SD,
           AVG(CASE WHEN estadoactividad = 'No presentado' THEN 1.0 ELSE 0.0 END) AS ACT_PCT_NOENTREGA,
           AVG(CASE WHEN ESTADO LIKE '%reprob%' THEN 1.0 ELSE 0.0 END)            AS ACT_PCT_REPROBADO,
           MAX(CAST(CREDITOS_INSCRITOS AS FLOAT)) AS ACT_CREDITOS
    FROM plataforma.permanencia WHERE cod_periodo IN ({lista})
    GROUP BY num_identificacion, cod_periodo''')

sub = base[base['PERIODO'].isin([p.strip() for p in PERIODOS])].copy()
sub = sub.merge(plat, on=['IDENTIFICACION', 'PERIODO'], how='left')
PLAT = [c for c in sub.columns if c.startswith('ACT_')]
for c in PLAT: sub[c] = pd.to_numeric(sub[c], errors='coerce')
PLAT = [c for c in PLAT if sub[c].notna().any()]

print(f"Cobertura de plataforma: {sub['ACT_N'].notna().mean()*100:.1f} %")
print(f"Cobertura de variables académicas: {sub['PORCENTAJE_APROBACION'].notna().mean()*100:.1f} %")

ptr = sub[sub['PERIODO'] == PERIODOS[0].strip()]
pte = sub[sub['PERIODO'] == PERIODOS[-1].strip()]
spw3 = (ptr['TARGET'] == 0).sum() / max((ptr['TARGET'] == 1).sum(), 1)
NUM_OK = [c for c in NUM if ptr[c].notna().any()]

for etiqueta, num, cat in [("A  variables administrativas", NUM_OK, CAT),
                           ("B  administrativas + plataforma", NUM_OK + PLAT, CAT),
                           ("C  solo plataforma", PLAT, [])]:
    mm = modelo(spw3, num, cat); mm.fit(ptr[num + cat], ptr['TARGET'])
    pp = mm.predict_proba(pte[num + cat])[:, 1]
    tramos, tasa = precision_en_tramo(pte['TARGET'], pp)
    print(f"{etiqueta:<34} AUC {roc_auc_score(pte['TARGET'], pp):.4f} | {tramos}")

In [ ]:
# ¿La señal anticipa el abandono, o lo está midiendo?
comp = sub.groupby('TARGET').agg(
    estudiantes=('ACT_N', 'size'), actividades=('ACT_N', 'mean'),
    cursos=('ACT_CURSOS', 'mean'), creditos=('ACT_CREDITOS', 'mean'),
    nota=('ACT_NOTA_MEDIA', 'mean')).round(2)
comp

**Resultado.** La actividad en plataforma aporta **+0.24 de AUC** y sola alcanza 0.838,
frente a 0.614 del resto de variables. Concentra el 69 % de la importancia del modelo
conjunto.

El patrón es nítido: quienes desertan se matriculan con **la misma carga** —igual número de
cursos y de créditos— pero generan **un 25 % menos de actividad registrada**.

**Y aun así no puede darse por bueno**, por tres razones:

1. **El baseline está incompleto**: las variables de rendimiento académico vienen nulas en la
   fuente para los periodos con cobertura de plataforma.
2. **No hay marca temporal por actividad.** Sin la fecha de cada registro no se puede
   distinguir *"se desenganchó en las primeras semanas"* —señal legítima— de *"dejó de
   participar al irse"*, que sería medir el desenlace.
3. **La cobertura histórica es de un solo año** y sus periodos son flujos paralelos por
   cohorte: menos del 1 % de los estudiantes aparece en dos consecutivos, así que no permite
   construir el desfase.

**Requisito para cerrar la pregunta: una columna de fecha por actividad.** Con ella se trunca
el histórico a las primeras semanas y se vuelve a medir.

---
## Resumen

| Sección | Afirmación | Estado |
|---|---|---|
| 1 | El target se recalculó en origen | **confirmada** |
| 2 | El preprocesamiento antes del split infla | descartada (−0.004) |
| 3 | La partición temporal infla | descartada con datos actuales (−0.006) |
| 4 | Anticiparse un periodo es viable | descartada (−0.128 de AUC) |
| 5 | La actividad en plataforma aporta | aporta, pero **no verificable** sin marca temporal |

Cuatro de las cinco hipótesis resultaron falsas. Se documentan junto a la confirmada porque
descartar una causa con una medición vale tanto como confirmarla.